In [1]:
import pyarrow.parquet as pq
import pandas as pd

# Load the posts data
posts_path = "../data/posting/cleaned/chunk_0_posts.parquet"
posts_table = pq.read_table(posts_path)

print("Posts data schema:")
print(posts_table.schema)
print(f"Total posts: {len(posts_table)}")
print("\nFirst few rows of posts:")
print(posts_table.to_pandas().head())

# Load the profiles data
profiles_path = "../data/posting/cleaned/profiles.parquet"
profiles_table = pq.read_table(profiles_path)

print("\nProfiles data schema:")
print(profiles_table.schema)
print(f"Total profiles: {len(profiles_table)}")
print("\nFirst few rows of profiles:")
print(profiles_table.to_pandas().head())

Posts data schema:
created_at: timestamp[us, tz=UTC]
did_id: int64
Total posts: 4994663

First few rows of posts:
                        created_at    did_id
0 2023-09-16 19:36:58.558000+00:00  31955122
1 2024-11-15 16:22:35.542000+00:00  31955122
2 2024-11-16 21:00:29.703000+00:00  31955122
3 2024-11-22 20:30:39.146000+00:00  31955122
4 2024-12-13 18:34:28.434000+00:00  31955122

Profiles data schema:
did_id: int64
created_at: timestamp[us, tz=UTC]
Total profiles: 27922675

First few rows of profiles:
   did_id                       created_at
0       1 2024-11-15 07:04:16.352000+00:00
1       2 2024-11-27 18:21:32.298000+00:00
2       3 2024-11-16 13:04:51.383000+00:00
3       4 2024-11-27 14:06:44.280000+00:00
4       5 2024-12-08 15:24:03.663000+00:00


## Filtering
Remove users with less than three posts. They aren't of interests. 

In [2]:
posts_df = posts_table.to_pandas()

print(f"Initial posts count: {len(posts_df)}")
print(f"Unique users: {posts_df['did_id'].nunique()}")

# Convert created_at to datetime if needed
if not pd.api.types.is_datetime64_any_dtype(posts_df['created_at']):
    posts_df['created_at'] = pd.to_datetime(posts_df['created_at'])

# Group by user and count posts
user_post_counts = posts_df.groupby('did_id').size().reset_index(name='post_count')

# Filter users with at least 3 posts
active_users = user_post_counts[user_post_counts['post_count'] >= 3]['did_id']
filtered_posts_df = posts_df[posts_df['did_id'].isin(active_users)]

print(f"Posts after filtering users with <3 posts: {len(filtered_posts_df)}")
print(f"Active users (≥3 posts): {len(active_users)}")

# Sort by user and timestamp for time series creation
filtered_posts_df = filtered_posts_df.sort_values(['did_id', 'created_at'])

Initial posts count: 4994663
Unique users: 130111
Posts after filtering users with <3 posts: 4924101
Active users (≥3 posts): 76262


## Merging and Processing
We want a table where for each user we have the following stuff: 
1) Join date timestamp. 
2) First post timestamp.
3) day_1_posts, day_2_posts, ..., day_n_posts.
4) TODO: Look at the block database. 
5) TODO: Look at the likes database.

In [3]:
# Get the users of interests from profiles.
unique_users = filtered_posts_df['did_id'].unique()
print(f"Processing {len(unique_users)} users...")

profiles_df = profiles_table.to_pandas()

active_profiles = profiles_df[
    profiles_df['did_id'].isin(active_users)
][['did_id', 'created_at']].rename(columns={'created_at': 'join_date'})

print(f"Active users with valid join dates: {len(active_profiles)}")

Processing 76262 users...
Active users with valid join dates: 52167


In [4]:
# Inner join profiles with posts
merged_df = filtered_posts_df.merge(
    active_profiles, on='did_id', how='inner'
)
print(merged_df.head())

                        created_at  did_id                        join_date
0 2024-08-30 21:25:28.002000+00:00     318 2024-08-30 21:04:26.303000+00:00
1 2024-08-30 22:29:13.143000+00:00     318 2024-08-30 21:04:26.303000+00:00
2 2024-08-31 14:01:03.295000+00:00     318 2024-08-30 21:04:26.303000+00:00
3 2024-08-31 14:23:46.432000+00:00     318 2024-08-30 21:04:26.303000+00:00
4 2024-08-31 14:39:12.549000+00:00     318 2024-08-30 21:04:26.303000+00:00


In [5]:
# Create a user_table with "did_id", "user_join_date" and "user_first_post"
user_first_post = merged_df.groupby('did_id')['created_at'].min().reset_index()
user_first_post = user_first_post.rename(columns={'created_at': 'first_post_date'})

user_table = active_profiles[['did_id', 'join_date']].merge(user_first_post, on='did_id', how='inner')
print("User Table")
print(user_table.head())

User Table
   did_id                        join_date                  first_post_date
0     318 2024-08-30 21:04:26.303000+00:00 2024-08-30 21:25:28.002000+00:00
1    1032 2024-10-19 12:08:26.894000+00:00 2024-11-17 23:04:06.860000+00:00
2    1738 2024-11-24 22:41:30.245000+00:00 2024-11-24 22:44:22.029000+00:00
3    3316 2024-06-08 03:54:37.834000+00:00 2024-06-08 06:02:30.972000+00:00
4    3360 2024-11-17 05:41:26.862000+00:00 2024-11-17 05:44:20.897000+00:00


In [6]:
# Calculate days since joining, filter the first month of activity, merge them to user_table

merged_df['days_since_join'] = (
    (merged_df['created_at'] - merged_df['join_date']).dt.total_seconds() / (24 * 3600)
).round().astype(int)

first_month_posts = merged_df[
    (merged_df['days_since_join'] >= 0) & 
    (merged_df['days_since_join'] <= 30)
]

daily_post_counts = first_month_posts.groupby(['did_id', 'days_since_join']).size().reset_index(name='post_count')

# Pivot
time_series_wide = daily_post_counts.pivot_table(
    index='did_id', 
    columns='days_since_join', 
    values='post_count', 
    fill_value=0
).reset_index()

# Rename the day-posts columns
time_series_wide.columns = ['did_id'] + [f'day_{int(col)}_posts' for col in time_series_wide.columns[1:]]

# Merge into final table
user_table = user_table.merge(time_series_wide, on='did_id', how='left')

print(user_table.head())

   did_id                        join_date                  first_post_date  \
0     318 2024-08-30 21:04:26.303000+00:00 2024-08-30 21:25:28.002000+00:00   
1    1032 2024-10-19 12:08:26.894000+00:00 2024-11-17 23:04:06.860000+00:00   
2    1738 2024-11-24 22:41:30.245000+00:00 2024-11-24 22:44:22.029000+00:00   
3    3316 2024-06-08 03:54:37.834000+00:00 2024-06-08 06:02:30.972000+00:00   
4    3360 2024-11-17 05:41:26.862000+00:00 2024-11-17 05:44:20.897000+00:00   

   day_0_posts  day_1_posts  day_2_posts  day_3_posts  day_4_posts  \
0          2.0          5.0          4.0          4.0          3.0   
1          0.0          0.0          0.0          0.0          0.0   
2         19.0          0.0          0.0          0.0          0.0   
3          5.0          0.0          0.0          1.0          0.0   
4          1.0         18.0         11.0          4.0         20.0   

   day_5_posts  day_6_posts  ...  day_21_posts  day_22_posts  day_23_posts  \
0          3.0          0.

# Saving 

In [7]:
# Save the processed data for future use
output_path = "../data/posting/processed/user_activity.parquet"
user_table.to_parquet(output_path, index=False)
print(f"\nData saved to: {output_path}")


Data saved to: ../data/posting/processed/user_activity.parquet
